<a href="https://colab.research.google.com/github/fauzilfebriwal/Project-Computer-Vision-Perbandingan-3-Model/blob/main/UAS_COMPUTER_VISION_241012050026_FAUZIL_FEBRIWAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CELL 1 — Install Library

In [ ]:
!pip install timm -q

CELL 2 — Import Library

In [ ]:
import torch
import timm
import random
import numpy as np
import pandas as pd

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader
from torch.utils.data import Subset

from sklearn.metrics import accuracy_score

CELL 3 — Cek GPU

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cpu


CELL 3 — Import Library

In [ ]:
import torch
import torchvision
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

CELL 4 — Download Dataset

In [ ]:
!git clone https://github.com/spMohanty/PlantVillage-Dataset.git

fatal: destination path 'PlantVillage-Dataset' already exists and is not an empty directory.


CELL 5 — Transform

In [ ]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

CELL 6 — Load Dataset

In [ ]:
dataset = datasets.ImageFolder(
    "/content/PlantVillage-Dataset/raw/color",
    transform=transform
)

print(len(dataset))
print(len(dataset.classes))

54305
38


CELL 7 — Ambil hanya 5000 gambar untuk project

In [ ]:
sample_size = 5000

indices = random.sample(
    range(len(dataset)),
    sample_size
)

dataset = Subset(
    dataset,
    indices
)

CELL 8 — Split Dataset

In [ ]:
train_size = int(
    0.8*len(dataset)
)

test_size = len(dataset)-train_size

train_dataset,test_dataset = \
torch.utils.data.random_split(
    dataset,
    [train_size,test_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

CELL 9 — Function Training

In [ ]:
def train_model(
    model,
    epochs=3
):

    model.to(device)

    criterion = \
        torch.nn.CrossEntropyLoss()

    optimizer = \
        torch.optim.Adam(
            model.parameters(),
            lr=0.0001
        )

    for epoch in range(epochs):

        model.train()

        total=0

        for x,y in train_loader:

            x=x.to(device)
            y=y.to(device)

            optimizer.zero_grad()

            pred=model(x)

            loss=criterion(
                pred,
                y
            )

            loss.backward()

            optimizer.step()

            total += loss.item()

        print(
            "Epoch",
            epoch+1,
            total
        )

    return model

CELL 10 — Evaluation

In [ ]:
def evaluate(model):

    model.eval()

    y_true=[]
    y_pred=[]

    with torch.no_grad():

        for x,y in test_loader:

            x=x.to(device)

            pred=model(x)

            pred=pred.argmax(1)

            y_true.extend(
                y.numpy()
            )

            y_pred.extend(
                pred.cpu().numpy()
            )

    acc = accuracy_score(
        y_true,
        y_pred
    )

    print(
        "Accuracy:",
        acc
    )

    return acc

RESNET50

In [ ]:
resnet = models.resnet50(
    weights='DEFAULT'
)

for p in resnet.parameters():
    p.requires_grad=False

resnet.fc = torch.nn.Linear(
    resnet.fc.in_features,
    38
)

resnet = train_model(
    resnet,
    epochs=3
)

resnet_acc = evaluate(
    resnet
)

Epoch 1 404.8725116252899
Epoch 2 348.730544090271
Epoch 3 309.04151701927185
Accuracy: 0.464


EFFICIENTNET

In [ ]:
efficient = \
timm.create_model(
    'efficientnet_b0',
    pretrained=True,
    num_classes=38
)

for p in efficient.parameters():
    p.requires_grad=False

efficient.classifier = \
torch.nn.Linear(
    efficient.classifier.in_features,
    38
)

efficient = train_model(
    efficient,
    epochs=3
)

efficient_acc = \
evaluate(efficient)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Epoch 1 407.0057952404022
Epoch 2 330.47829842567444
Epoch 3 276.2041153907776
Accuracy: 0.664


VISION TRANSFORMER

In [ ]:
vit = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True,
    num_classes=38
)

for p in vit.parameters():
    p.requires_grad=False

vit.head = torch.nn.Linear(
    vit.head.in_features,
    38
)

vit = train_model(
    vit,
    epochs=2
)

vit_acc = evaluate(vit)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Epoch 1 328.8470116853714
Epoch 2 171.0333452820778
Accuracy: 0.791


HASIL

In [ ]:
result = pd.DataFrame({

    'Model':[
        'ResNet50',
        'EfficientNet',
        'ViT'
    ],

    'Accuracy':[
        resnet_acc,
        efficient_acc,
        vit_acc
    ]
})

result

,Model,Accuracy
0,ResNet50,0.464
1,EfficientNet,0.664
2,ViT,0.791
